# Week 2 — Inflation Data Fix

The original inflation data from the IMF IFS database only covered 52 countries, instead
- WEO (World Economic Outlook) annual CPI data is used, which covers 196 countries, and interpolates it to monthly values.

- 36 territories (like British Virgin Islands and Cocos Islands) that have no exchange rate or GDP data are removed.



In [1]:
import pandas as pd
from pathlib import Path

def project_root() -> Path:
    marker = Path("data/final/master_trade_flow.csv")
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / marker).is_file():
            return d
    return Path("/Users/angollapraveengoud/Trade_Flow_project")

base = project_root()

master = pd.read_csv(base / "data/final/master_trade_flow.csv")
inf = pd.read_csv(base / "data/cleaned/inflation_cleaned.csv")
ir = pd.read_csv(base / "data/cleaned/interest_rate_cleaned.csv")

master_names = set(master["country_imf"].unique())
inf_names = set(inf["country"].unique())
ir_names = set(ir["country"].unique())

inf_unmatched = sorted(inf_names - master_names)
print(f"Inflation has {len(inf_names)} countries")
print(f"Unmatched: {len(inf_unmatched)}")
print()
for name in inf_unmatched:
    print(f"  {name}")

Inflation has 196 countries
Unmatched: 6

  Côte D'Ivoire
  Puerto Rico
  São Tomé And Príncipe, Democratic Republic Of
  Türkiye, Republic Of
  United States
  Venezuela, República Bolivariana De


In [2]:
# Find what master calls these countries
search_terms = ["Croatia", "Estonia", "Latvia", "Lithuania", "Slovak", "Slovenia", "West Bank"]

for term in search_terms:
    matches = [n for n in master_names if term.lower() in n.lower()]
    print(f"{term}: {matches}")

Croatia: ['Croatia']
Estonia: ['Estonia']
Latvia: ['Latvia']
Lithuania: ['Lithuania']
Slovak: ['Slovakia']
Slovenia: ['Slovenia']
West Bank: ['West Bank Administered By Israel']


In [3]:
inf_to_master = {
    "Croatia, Republic Of": "Croatia",
    "Estonia, Republic Of": "Estonia",
    "Latvia, Republic Of": "Latvia",
    "Lithuania, Republic Of": "Lithuania",
    "Slovak Republic": "Slovakia",
    "Slovenia, Republic Of": "Slovenia",
    "West Bank And Gaza": "West Bank Administered By Israel",
}

inf["country"] = inf["country"].replace(inf_to_master)

# Checking Fix
inf_unmatched_after = sorted(set(inf["country"].unique()) - master_names)
print(f"Inflation unmatched BEFORE: 7")
print(f"Inflation unmatched AFTER: {len(inf_unmatched_after)}")

Inflation unmatched BEFORE: 7
Inflation unmatched AFTER: 6


In [4]:
ir_unmatched = sorted(ir_names - master_names)
print(f"Interest rate has {len(ir_names)} countries")
print(f"Unmatched: {len(ir_unmatched)}")
print()
for name in ir_unmatched:
    print(f"  {name}")

Interest rate has 133 countries
Unmatched: 5

  Croatia, Republic Of
  Eastern Caribbean Currency Union (Eccu)
  United States
  West African Economic And Monetary Union (Waemu)
  West Bank And Gaza


In [5]:
ir_to_master = {
    "Croatia, Republic Of": "Croatia",
    "West Bank And Gaza": "West Bank Administered By Israel",
}

ir["country"] = ir["country"].replace(ir_to_master)

ir_unmatched_after = sorted(set(ir["country"].unique()) - master_names)
# Removing known non-countries
non_countries = {"Eastern Caribbean Currency Union (Eccu)",
                 "United States",
                 "West African Economic And Monetary Union (Waemu)"}
real_unmatched = [n for n in ir_unmatched_after if n not in non_countries]

print(f"Interest rate unmatched BEFORE: 5")
print(f"Real unmatched AFTER: {len(real_unmatched)}")

Interest rate unmatched BEFORE: 5
Real unmatched AFTER: 0


## Step 1 — Re-attach Inflation with Fixed Country Names

First attempt: re-merge using the corrected country names from above.
The WEO data rebuild happens next (cells below) and produces the actual final version.
Interest rate is intentionally not merged — coverage is too sparse (~56% missing) to be useful in modeling.


In [6]:
# INTERMEDIATE STEP - merged using the IFS-based inflation with fixed country names.
master = pd.read_csv(base / "data/final/master_trade_flow.csv")
master = master.drop(columns=["inflation", "interest_rate"], errors="ignore")

master = master.merge(inf, left_on=["country_imf", "year", "month"],
                      right_on=["country", "year", "month"],
                      how="left").drop(columns=["country"])

print("Shape:", master.shape)
print()
print("Inflation missing % after merge")
print(f"inflation: {round(100 * master['inflation'].isna().mean(), 1)}% missing")
print("(interest_rate is intentionally not merged — see note above.)")


Shape: (22306, 12)

Inflation missing % after merge
inflation: 18.5% missing
(interest_rate is intentionally not merged — see note above.)


In [7]:
top30 = master.groupby("country_imf")[["exports_usd", "imports_usd"]].sum().sum(axis=1).sort_values(ascending=False).head(30).index

top30_inf = master[master["country_imf"].isin(top30)].groupby("country_imf")["inflation"].apply(lambda x: round(100 * x.isna().mean())).sort_index()

if "interest_rate" in master.columns:
    top30_ir = master[master["country_imf"].isin(top30)].groupby("country_imf")["interest_rate"].apply(lambda x: round(100 * x.isna().mean())).sort_index()
    compare = pd.DataFrame({"inflation_null%": top30_inf, "interest_rate_null%": top30_ir})
    print("TOP 30 PARTNERS — NULL %")
    print(compare.to_string())
    print()
    print(f"Top 30 with inflation data (< 100%): {(compare['inflation_null%'] < 100).sum()} of 30")
    print(f"Top 30 with interest rate data (< 100%): {(compare['interest_rate_null%'] < 100).sum()} of 30")
else:
    compare = pd.DataFrame({"inflation_null%": top30_inf})
    print("=TOP 30 PARTNERS — NULL % (inflation only; interest_rate not in master)")
    print(compare.to_string())
    print()
    print(f"Top 30 with inflation data (< 100%): {(compare['inflation_null%'] < 100).sum()} of 30")


=TOP 30 PARTNERS — NULL % (inflation only; interest_rate not in master)
                                                                     inflation_null%
country_imf                                                                         
Australia                                                                          0
Belgium                                                                            0
Brazil                                                                             0
Canada                                                                             0
Chile                                                                              0
China, People'S Republic Of                                                        0
Colombia                                                                           0
France                                                                             0
Germany                                                                       

## Step 2 — Rebuild Inflation from WEO Data

The IFS database was missing most major economies (China, Japan, Canada, etc.).
The WEO file (`inflation_data.csv`) has the "All Items, CPI, Period average, percent change"
indicator for 210 countries — including all of the top 30 trade partners.
The cells below extract this, clean country names, and produce a proper monthly inflation series.

In [8]:
ifs_path = base / "data/raw/inflation_monthly_raw.csv"
weo_path = base / "data/raw/inflation_data.csv"

if ifs_path.is_file():
    inf_raw = pd.read_csv(ifs_path, encoding="latin1", low_memory=False, on_bad_lines="skip")
    print("Loaded IFS-style raw:", ifs_path.name, "| shape:", inf_raw.shape)
elif weo_path.is_file():
    inf_raw = pd.read_csv(weo_path, encoding="utf-8-sig", low_memory=False, on_bad_lines="skip")
    print("Loaded WEO-style raw:", weo_path.name, "| shape:", inf_raw.shape)

if "COUNTRY" not in inf_raw.columns and "country" in inf_raw.columns:
    inf_raw = inf_raw.rename(columns={"country": "COUNTRY"})

big_partners = ["China", "Japan", "Canada", "Brazil", "India", "Australia",
                "Korea", "Mexico", "Indonesia", "Malaysia"]

for partner in big_partners:
    sub = inf_raw[inf_raw["COUNTRY"].astype(str).str.contains(partner, case=False, na=False)]
    print(f"\n{partner} ({len(sub)} rows):")
    if "INDEX_TYPE" in sub.columns:
      print(f"  INDEX_TYPE: {sub['INDEX_TYPE'].unique().tolist()[:8]}")
    if "TYPE_OF_TRANSFORMATION" in sub.columns:
      print(f"  TYPE_OF_TRANSFORMATION: {sub['TYPE_OF_TRANSFORMATION'].unique().tolist()[:8]}")
    if "COICOP_1999" in sub.columns:
      print(f"  COICOP_1999: {sub['COICOP_1999'].unique().tolist()[:5]}")
    if "INDICATOR" in sub.columns:
      print(f"  INDICATOR (sample): {sub['INDICATOR'].unique().tolist()[:5]}")
    if "FREQUENCY" in sub.columns:
      print(f"  FREQUENCY: {sub['FREQUENCY'].unique().tolist()[:5]}")


Loaded WEO-style raw: inflation_data.csv | shape: (8208, 116)

China (156 rows):
  INDEX_TYPE: [nan, 'Consumer price index (CPI)', 'Price deflator']
  COICOP_1999: [nan]
  INDICATOR (sample): ['Unemployment rate', 'Gross debt, General government, Domestic currency', 'All Items, Consumer price index (CPI), Period average, percent change', 'Gross domestic product (GDP), Constant prices, Percent change', 'Employed persons, Persons for countries / Index for country groups']
  FREQUENCY: ['Annual']

Japan (44 rows):
  INDEX_TYPE: [nan, 'Consumer price index (CPI)', 'Price deflator']
  COICOP_1999: [nan]
  INDICATOR (sample): ['Net lending (+) / net borrowing (-), General government, Domestic currency', 'Primary net lending (+) / net borrowing (-), General government, Percent of GDP', 'All Items, Consumer price index (CPI), Period average', 'Gross capital formation, Percent of GDP', 'Net lending (+) / net borrowing (-), General government, Percent of GDP']
  FREQUENCY: ['Annual']

Canada (44

In [9]:
# Checking cleaned file (long format) for China spellings
inf_check = pd.read_csv(base / "data/cleaned/inflation_cleaned.csv")
china_names = [n for n in inf_check["country"].unique() if "china" in str(n).lower()]
print("China in inflation_cleaned.csv:", china_names)


China in inflation_cleaned.csv: ['Taiwan Province Of China', "Hong Kong Special Administrative Region, People'S Republic Of China", "Macao Special Administrative Region, People'S Republic Of China", "China, People'S Republic Of"]


In [10]:
# Overwrite cleaned files with fixed names
inf.to_csv(base / "data/cleaned/inflation_cleaned.csv", index=False)
ir.to_csv(base / "data/cleaned/interest_rate_cleaned.csv", index=False)


In [11]:
# INTERMEDIATE STEP — full re-merge with fixed names.
exp = pd.read_csv(base / "data/cleaned/exports_cleaned.csv")
imp = pd.read_csv(base / "data/cleaned/imports_cleaned.csv")
er = pd.read_csv(base / "data/cleaned/exchange_rate_cleaned.csv")
gdp = pd.read_csv(base / "data/cleaned/gdp_cleaned.csv")
inf = pd.read_csv(base / "data/cleaned/inflation_cleaned.csv")

# Loading country mapping
mapping = pd.read_csv(base / "data/country_name_mapping.csv")
trade_to_imf = dict(zip(mapping["trade_name"], mapping["imf_name"]))

# Applying mapping to exports/imports
exp["country_imf"] = exp["country"].map(trade_to_imf).fillna(exp["country"])
imp["country_imf"] = imp["country"].map(trade_to_imf).fillna(imp["country"])

# Expanding GDP to monthly
gdp_m = gdp.merge(pd.DataFrame({"month": range(1, 13)}), how="cross")

# Merging all
m = exp.merge(imp, on=["country_imf", "year", "month"], how="outer", suffixes=("_exp", "_imp"))
m = m.merge(er, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])
m = m.merge(gdp_m, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])
m = m.merge(inf, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])


master = m[(m["year"] >= 2017) & (m["year"] <= 2024)].copy()

print("Master shape:", master.shape)
print()
print("NULL %")
print((master.isna().mean() * 100).round(1).sort_values(ascending=False))


Master shape: (22306, 12)

NULL %
gdp_billions        20.2
inflation           18.5
exchange_rate       14.8
country_exp          0.6
country_code_exp     0.6
exports_usd          0.6
country_imp          0.2
country_code_imp     0.2
imports_usd          0.2
year                 0.0
month                0.0
country_imf          0.0
dtype: float64


In [12]:
master.to_csv(base / "data/final/master_trade_flow.csv", index=False)
print("Top 30 inflation coverage:",
      (master[master["country_imf"].isin(master.groupby("country_imf")[["exports_usd","imports_usd"]].sum().sum(axis=1).sort_values(ascending=False).head(30).index)]
       .groupby("country_imf")["inflation"].apply(lambda x: x.notna().any()).sum()), "of 30")

Top 30 inflation coverage: 29 of 30


In [13]:
weo = pd.read_csv(base / "data/raw/imf_res_weo_gdp_wide.csv", encoding="latin1", low_memory=False)

print("Columns:", weo.columns.tolist()[:10])
print()
print("INDICATOR values:")
if "INDICATOR" in weo.columns:
    print(weo["INDICATOR"].unique().tolist())
else:
    # Checking what columns exist
    print(weo.head(2).to_string())

Columns: ['ï»¿"DATASET"', 'SERIES_CODE', 'OBS_MEASURE', 'COUNTRY', 'INDICATOR', 'FREQUENCY', 'SCALE', '2017', '2018', '2019']

INDICATOR values:
['Gross domestic product (GDP), Current prices, US dollar']


In [14]:
inf_new = pd.read_csv(base / "data/raw/inflation_data.csv", encoding="utf-8-sig", low_memory=False, on_bad_lines="skip")

print("Shape:", inf_new.shape)
print("Columns:", inf_new.columns.tolist()[:10])
print()

# Checking if China, Japan, Canada have actual values
for partner in ["China", "Japan", "Canada", "Mexico", "Brazil", "India", "Australia", "Korea"]:
    sub = inf_new[inf_new["COUNTRY"].str.contains(partner, case=False, na=False)]
    if len(sub) == 0:
        print(f"{partner}: NOT FOUND")
    else:
        val_2023 = sub["2023"].values[0] if "2023" in sub.columns else "no 2023 col"
        print(f"{partner}: COUNTRY='{sub['COUNTRY'].iloc[0]}' | 2023 value={val_2023}")

Shape: (8208, 116)
Columns: ['data?"DATASET"', 'SERIES_CODE', 'OBS_MEASURE', 'COUNTRY', 'INDICATOR', 'FREQUENCY', 'SCALE', 'DECIMALS_DISPLAYED', 'FUNCTIONAL_CAT', 'INT_ACC_ITEM']

China: COUNTRY='Taiwan Province of China' | 2023 value=3.48
Japan: COUNTRY='Japan' | 2023 value=-13354.126
Canada: COUNTRY='Canada' | 2023 value=20.341
Mexico: COUNTRY='Mexico' | 2023 value=477.321
Brazil: COUNTRY='Brazil' | 2023 value=44630608935457.0
India: COUNTRY='India' | 2023 value=61734.832
Australia: COUNTRY='Australia' | 2023 value=14.019
Korea: COUNTRY='Korea, Republic of' | 2023 value=23.083


In [15]:
print("Unique indicators:")
for ind in inf_new["INDICATOR"].unique():
    print(f"  {ind}")

Unique indicators:
  Employed persons, Persons for countries / Index for country groups
  Gross domestic product (GDP), Current prices, Per capita, Domestic currency
  Population, Persons for countries / Index for country groups
  Gross domestic product (GDP), Constant prices, Domestic currency
  Current account balance (credit less debit), Percent of GDP
  Financial account balance (assets less liabilities), US dollar
  Net lending (+) / net borrowing (-), General government, Percent of GDP
  Gross domestic product (GDP), Price deflator, Index
  Expenditure, General government, Percent of GDP
  Net debt, General government, Domestic currency
  Revenue, General government, Percent of GDP
  Gross domestic product (GDP), Per capita, purchasing power parity (PPP) international dollar, ICP benchmarks 2017-2021
  Current account balance (credit less debit), US dollar
  Expenditure, General government, Domestic currency
  Gross domestic product (GDP), Constant prices, Per capita, Domestic cu

In [16]:
inf_cpi = inf_new[inf_new["INDICATOR"] == "All Items, Consumer price index (CPI), Period average, percent change"].copy()

print(f"Rows after filter: {len(inf_cpi)}")
print()

for partner in ["China", "Japan", "Canada", "Mexico", "Brazil", "India", "Australia", "Korea", "Singapore", "Saudi"]:
    sub = inf_cpi[inf_cpi["COUNTRY"].str.contains(partner, case=False, na=False)]
    if len(sub) == 0:
        print(f"{partner}: NOT FOUND")
    else:
        val = sub["2023"].values[0]
        print(f"{partner}: '{sub['COUNTRY'].iloc[0]}' | 2023={val}")

Rows after filter: 210

China: 'Taiwan Province of China' | 2023=2.492
Japan: 'Japan' | 2023=3.268
Canada: 'Canada' | 2023=3.885
Mexico: 'Mexico' | 2023=5.526
Brazil: 'Brazil' | 2023=4.594
India: 'India' | 2023=5.361
Australia: 'Australia' | 2023=5.594
Korea: 'Korea, Republic of' | 2023=3.597
Singapore: 'Singapore' | 2023=4.834
Saudi: 'Saudi Arabia' | 2023=2.327


In [17]:
china_rows = inf_cpi[inf_cpi["COUNTRY"].str.contains("China", case=False, na=False)]
print("All China matches:")
for _, row in china_rows.iterrows():
    print(f"  '{row['COUNTRY']}' | 2023={row['2023']}")

All China matches:
  'Taiwan Province of China' | 2023=2.492
  'Hong Kong Special Administrative Region, People's Republic of China' | 2023=2.097
  'Macao Special Administrative Region, People's Republic of China' | 2023=0.939
  'China, People's Republic of' | 2023=0.228


In [18]:
# Filtering out aggregate/regional rows
agg_pattern = r"Advanced Economies|^G7$|Euro Area|^World$|ASEAN-5|European Union|Emerging Market|Latin America|Middle East|Sub-Saharan|Emerging and Developing|Other Advanced"
inf_clean = inf_cpi[~inf_cpi["COUNTRY"].str.contains(agg_pattern, case=False, na=False)].copy()

# Keeping only country + year columns
year_cols = [str(y) for y in range(2017, 2025)]
inf_clean = inf_clean[["COUNTRY"] + year_cols].copy()

# Wide to long conversion
inf_long = inf_clean.melt(id_vars=["COUNTRY"], var_name="year", value_name="inflation")
inf_long["year"] = pd.to_numeric(inf_long["year"])
inf_long["inflation"] = pd.to_numeric(inf_long["inflation"], errors="coerce")
inf_long = inf_long.dropna(subset=["inflation"])
inf_long = inf_long.rename(columns={"COUNTRY": "country"})

# Spreading to 12 months (same as GDP)
inf_monthly = inf_long.merge(pd.DataFrame({"month": range(1, 13)}), how="cross")

print("Shape:", inf_monthly.shape)
print("Countries:", inf_monthly["country"].nunique())
print()
print("Sample:")
print(inf_monthly[inf_monthly["country"].str.contains("China, People")].head(3).to_string(index=False))

Shape: (18756, 4)
Countries: 196

Sample:
                    country  year  inflation  month
China, People's Republic of  2017      1.569      1
China, People's Republic of  2017      1.569      2
China, People's Republic of  2017      1.569      3


In [19]:
master_names = set(pd.read_csv(base / "data/final/master_trade_flow.csv")["country_imf"].unique())
inf_names = set(inf_monthly["country"].unique())

unmatched = sorted(inf_names - master_names)
print(f"Inflation countries: {len(inf_names)}")
print(f"Unmatched: {len(unmatched)}")
print()
for name in unmatched[:20]:
    print(f"  {name}")
if len(unmatched) > 20:
    print(f"  ... and {len(unmatched) - 20} more")

Inflation countries: 196
Unmatched: 64

  Afghanistan, Islamic Republic of
  Andorra, Principality of
  Antigua and Barbuda
  Armenia, Republic of
  Aruba, Kingdom of the Netherlands
  Azerbaijan, Republic of
  Bahrain, Kingdom of
  Belarus, Republic of
  Bosnia and Herzegovina
  China, People's Republic of
  Comoros, Union of the
  Congo, Democratic Republic of the
  Congo, Republic of
  Croatia, Republic of
  Côte d'Ivoire
  Egypt, Arab Republic of
  Equatorial Guinea, Republic of
  Eritrea, The State of
  Estonia, Republic of
  Eswatini, Kingdom of
  ... and 44 more


In [20]:
inf_monthly["country"] = inf_monthly["country"].str.title()

unmatched_after = sorted(set(inf_monthly["country"].unique()) - master_names)
print(f"Unmatched after str.title(): {len(unmatched_after)}")
print()
for name in unmatched_after:
    print(f"  {name}")

Unmatched after str.title(): 15

  Croatia, Republic Of
  Côte D'Ivoire
  Estonia, Republic Of
  Latvia, Republic Of
  Lithuania, Republic Of
  Marshall Islands, Republic Of The
  Netherlands, The
  Puerto Rico
  Slovak Republic
  Slovenia, Republic Of
  São Tomé And Príncipe, Democratic Republic Of
  Türkiye, Republic Of
  United States
  Venezuela, República Bolivariana De
  West Bank And Gaza


In [21]:
# Checking the new mappingsfor term in ["Marshall", "Netherlands", "Puerto Rico"]:
matches = [n for n in master_names if term.lower() in n.lower()]
print(f"{term}: {matches}")

West Bank: ['West Bank Administered By Israel']


In [22]:
inf_name_fix = {
    "Croatia, Republic Of": "Croatia",
    "Estonia, Republic Of": "Estonia",
    "Latvia, Republic Of": "Latvia",
    "Lithuania, Republic Of": "Lithuania",
    "Slovak Republic": "Slovakia",
    "Slovenia, Republic Of": "Slovenia",
    "West Bank And Gaza": "West Bank Administered By Israel",
    "Marshall Islands, Republic Of The": "Marshall Islands",
    "Netherlands, The": "Netherlands",
}
# Puerto Rico and United States are not trade partners

inf_monthly["country"] = inf_monthly["country"].replace(inf_name_fix)

unmatched_final = sorted(set(inf_monthly["country"].unique()) - master_names - {"Puerto Rico", "United States"})
print(f"Real unmatched after fix: {len(unmatched_final)}")

# Save
inf_monthly.to_csv(base / "data/cleaned/inflation_cleaned.csv", index=False)


Real unmatched after fix: 4


In [23]:
# Loading master, drop old inflation column if present
master = pd.read_csv(base / "data/final/master_trade_flow.csv")
master = master.drop(columns=["inflation"], errors="ignore")

# Loading new inflation
inf_new_clean = pd.read_csv(base / "data/cleaned/inflation_cleaned.csv")

# Merging
master = master.merge(inf_new_clean, left_on=["country_imf", "year", "month"],
                      right_on=["country", "year", "month"],
                      how="left").drop(columns=["country"])

# Save
master.to_csv(base / "data/final/master_trade_flow.csv", index=False)

print("Master shape:", master.shape)
print()
print("NULL % ")
print((master.isna().mean() * 100).round(1).sort_values(ascending=False))
print()

# Top 30 check
top30 = master.groupby("country_imf")[["exports_usd", "imports_usd"]].sum().sum(axis=1).sort_values(ascending=False).head(30).index
top30_inf = master[master["country_imf"].isin(top30)].groupby("country_imf")["inflation"].apply(lambda x: round(100 * x.isna().mean())).sort_index()
print("TOP 30 INFLATION NULL %")
print(top30_inf.to_string())
print()
print(f"Top 30 with inflation data: {(top30_inf < 100).sum()} of 30")

Master shape: (22306, 12)

NULL % 
gdp_billions        20.2
inflation           18.5
exchange_rate       14.8
country_exp          0.6
country_code_exp     0.6
exports_usd          0.6
country_imp          0.2
country_code_imp     0.2
imports_usd          0.2
year                 0.0
month                0.0
country_imf          0.0
dtype: float64

TOP 30 INFLATION NULL %
country_imf
Australia                                                                0
Belgium                                                                  0
Brazil                                                                   0
Canada                                                                   0
Chile                                                                    0
China, People'S Republic Of                                              0
Colombia                                                                 0
France                                                                   0
Germany      

In [24]:
# Check if WEO has any interest rate indicator
rate_indicators = [ind for ind in inf_new["INDICATOR"].unique() if "rate" in ind.lower() or "interest" in ind.lower()]
print("Rate-related indicators in WEO:")
for ind in rate_indicators:
    print(f"  {ind}")

Rate-related indicators in WEO:
  Rate, Domestic currency per international dollar in PPP terms, ICP benchmarks 2017-2021
  Unemployment rate
  External debt: total debt service, interest, US dollar
  External debt: total debt service, interest, Percent of exports of goods and services
  External debt: total debt service, interest, Percent of GDP


## Fix 1: Remove Territories with No Macro Data

Professor's concern: territories like New Caledonia have no exchange rate, GDP, or inflation — including them forces imputation or inflates missing data. Solution: exclude them from master.

In [25]:
master = pd.read_csv(base / "data/final/master_trade_flow.csv")

# Countries missing BOTH exchange_rate AND gdp_billions for ALL their rows
country_coverage = master.groupby("country_imf").agg(
    er_pct=("exchange_rate", lambda x: x.notna().mean()),
    gdp_pct=("gdp_billions", lambda x: x.notna().mean()),
    inf_pct=("inflation", lambda x: x.notna().mean()),
    rows=("year", "count")
).reset_index()

# Territories = no exchange rate AND no GDP
territories = country_coverage[(country_coverage["er_pct"] == 0) & (country_coverage["gdp_pct"] == 0)]

print(f"Total countries in master: {master['country_imf'].nunique()}")
print(f"Territories with NO ER and NO GDP: {len(territories)}")
print(f"Rows they occupy: {territories['rows'].sum()}")
print()
print("These will be removed:")
for _, row in territories.iterrows():
    print(f"  {row['country_imf']} ({int(row['rows'])} rows)")

Total countries in master: 234
Territories with NO ER and NO GDP: 30
Rows they occupy: 2722

These will be removed:
  British Indian Ocean Territories (96 rows)
  British Virgin Islands (96 rows)
  Christmas Island (96 rows)
  Cocos (Keeling) Islands (96 rows)
  Cook Islands (96 rows)
  Cuba (96 rows)
  Falkland Islands (Islas Malvinas) (96 rows)
  French Guiana (96 rows)
  French Southern And Antarctic Lands (96 rows)
  Gaza Strip Administered By Israel (94 rows)
  Guadeloupe (96 rows)
  Heard And Mcdonald Islands (93 rows)
  Korea, North (17 rows)
  Marshall Islands (96 rows)
  Martinique (96 rows)
  Mayotte (96 rows)
  Monaco (96 rows)
  Niue (96 rows)
  Norfolk Island (91 rows)
  Pitcairn Islands (95 rows)
  Reunion (96 rows)
  St Helena (96 rows)
  St Pierre And Miquelon (92 rows)
  Svalbard, Jan Mayen Island (93 rows)
  Tokelau (96 rows)
  Turks And Caicos Islands (96 rows)
  Vatican City (96 rows)
  Wallis And Futuna (93 rows)
  West Bank Administered By Israel (96 rows)
  Weste

In [26]:
remove_countries = territories["country_imf"].tolist()
master_clean = master[~master["country_imf"].isin(remove_countries)].copy()

print(f"Before: {master.shape[0]} rows, {master['country_imf'].nunique()} countries")
print(f"After:  {master_clean.shape[0]} rows, {master_clean['country_imf'].nunique()} countries")
print(f"Removed: {len(remove_countries)} territories, {master.shape[0] - master_clean.shape[0]} rows")
print()
print("NULL % AFTER TERRITORY REMOVAL")
print((master_clean.isna().mean() * 100).round(1).sort_values(ascending=False))

Before: 22306 rows, 234 countries
After:  19584 rows, 204 countries
Removed: 30 territories, 2722 rows

NULL % AFTER TERRITORY REMOVAL
gdp_billions        9.1
inflation           8.1
exchange_rate       2.9
country_exp         0.0
country_code_exp    0.0
exports_usd         0.0
year                0.0
month               0.0
country_imf         0.0
country_imp         0.0
country_code_imp    0.0
imports_usd         0.0
dtype: float64


In [27]:
master_clean.to_csv(base / "data/final/master_trade_flow.csv", index=False)
print("Countries:", master_clean["country_imf"].nunique())

Countries: 204


## Fix 2: Interpolate GDP & Inflation (Annual → Monthly)

Instead of copying the same annual value 12 times (fake monthly), linearly interpolate between years so each month has a slightly different value. This creates realistic monthly variation.

In [28]:
master = pd.read_csv(base / "data/final/master_trade_flow.csv")

# Remastered panel uses year + month only; interpolation
if "date" not in master.columns:
    master["date"] = pd.to_datetime(
        master["year"].astype(int).astype(str)
        + "-"
        + master["month"].astype(int).astype(str).str.zfill(2)
        + "-01"
    )
else:
    master["date"] = pd.to_datetime(master["date"])

print("BEFORE — Argentina GDP sample:")
arg = master[master["country_imf"] == "Argentina"].sort_values("date")
print(arg[["date", "gdp_billions"]].head(12).to_string(index=False))

# Interpolating GDP and inflation per country over time
for col in ["gdp_billions", "inflation"]:
    master[col] = master.sort_values(["country_imf", "date"]).groupby("country_imf")[col].transform(
        lambda x: x.interpolate(method="linear")
    )

print("\nAFTER — Argentina GDP sample:")
arg = master[master["country_imf"] == "Argentina"].sort_values("date")
print(arg[["date", "gdp_billions"]].head(12).to_string(index=False))


BEFORE — Argentina GDP sample:
      date  gdp_billions
2017-01-01       643.861
2017-02-01       643.861
2017-03-01       643.861
2017-04-01       643.861
2017-05-01       643.861
2017-06-01       643.861
2017-07-01       643.861
2017-08-01       643.861
2017-09-01       643.861
2017-10-01       643.861
2017-11-01       643.861
2017-12-01       643.861

AFTER — Argentina GDP sample:
      date  gdp_billions
2017-01-01       643.861
2017-02-01       643.861
2017-03-01       643.861
2017-04-01       643.861
2017-05-01       643.861
2017-06-01       643.861
2017-07-01       643.861
2017-08-01       643.861
2017-09-01       643.861
2017-10-01       643.861
2017-11-01       643.861
2017-12-01       643.861


In [29]:
for col in ["gdp_billions", "inflation"]:
    # Keeping only July (mid-year anchor), setting rest to NaN
    mask_not_july = master["month"] != 7
    original = master[col].copy()
    master.loc[mask_not_july, col] = float("nan")

    # Interpolating within each country
    master[col] = master.sort_values(["country_imf", "date"]).groupby("country_imf")[col].transform(
        lambda x: x.interpolate(method="linear")
    )

    # Filling any remaining NaN at edges (Jan-Jun of first year, Aug-Dec of last year)
    master[col] = master.groupby("country_imf")[col].transform(
        lambda x: x.ffill().bfill()
    )

print("AFTER interpolation — Argentina GDP:")
arg = master[master["country_imf"] == "Argentina"].sort_values("date")
print(arg[["date", "gdp_billions"]].head(24).to_string(index=False))

AFTER interpolation — Argentina GDP:
      date  gdp_billions
2017-01-01    643.861000
2017-02-01    643.861000
2017-03-01    643.861000
2017-04-01    643.861000
2017-05-01    643.861000
2017-06-01    643.861000
2017-07-01    643.861000
2017-08-01    633.908500
2017-09-01    623.956000
2017-10-01    614.003500
2017-11-01    604.051000
2017-12-01    594.098500
2018-01-01    584.146000
2018-02-01    574.193500
2018-03-01    564.241000
2018-04-01    554.288500
2018-05-01    544.336000
2018-06-01    534.383500
2018-07-01    524.431000
2018-08-01    517.958583
2018-09-01    511.486167
2018-10-01    505.013750
2018-11-01    498.541333
2018-12-01    492.068917


In [30]:
master.to_csv(base / "data/final/master_trade_flow.csv", index=False)
print("NULL % FINAL")
print((master.isna().mean() * 100).round(1).sort_values(ascending=False))

NULL % FINAL
gdp_billions        8.8
inflation           7.8
exchange_rate       2.9
country_exp         0.0
country_code_exp    0.0
exports_usd         0.0
year                0.0
month               0.0
country_imf         0.0
country_imp         0.0
country_code_imp    0.0
imports_usd         0.0
date                0.0
dtype: float64


In [31]:
master = pd.read_csv(base / "data/final/master_trade_flow.csv")
master = master.drop(columns=["interest_rate"], errors="ignore")

master.to_csv(base / "data/final/master_trade_flow.csv", index=False)
print("Dropped interest_rate column (if it was present)")
print("Shape:", master.shape)
print()
print("FINAL NULL %")
print((master.isna().mean() * 100).round(1).sort_values(ascending=False))

Dropped interest_rate column (if it was present)
Shape: (19584, 13)

FINAL NULL %
gdp_billions        8.8
inflation           7.8
exchange_rate       2.9
country_exp         0.0
country_code_exp    0.0
exports_usd         0.0
year                0.0
month               0.0
country_imf         0.0
country_imp         0.0
country_code_imp    0.0
imports_usd         0.0
date                0.0
dtype: float64
